# Transformer 架构从零实现教学

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/transformer-tutorial/blob/main/transformer_tutorial.ipynb)

本 notebook 基于 PyTorch 从零实现 Transformer 的每一个核心组件，对应论文：
> **Attention Is All You Need** (Vaswani et al., 2017)

## 目录
1. 环境与依赖
2. 输入嵌入 (Input Embedding)
3. 位置编码 (Positional Encoding)
4. 缩放点积注意力 (Scaled Dot-Product Attention)
5. 多头注意力 (Multi-Head Attention)
6. 前馈网络 (Feed-Forward Network)
7. 残差连接 + 层归一化 (Add & Norm)
8. 编码器层 (Encoder Layer)
9. 解码器层 (Decoder Layer)
10. 完整 Transformer 模型
11. 训练示例：数字序列翻转任务
12. 注意力权重可视化

## 1. 环境与依赖

In [ ]:
import math
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from torch.utils.data import Dataset, DataLoader

# 固定随机种子
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {DEVICE}')
print(f'PyTorch 版本: {torch.__version__}')

## 2. 输入嵌入 (Input Embedding)

将 token id 映射为 `d_model` 维的稠密向量，并按论文规定乘以 $\sqrt{d_{model}}$ 放大嵌入值。

In [ ]:
class InputEmbedding(nn.Module):
    """
    token_id -> d_model 维向量
    按论文乘以 sqrt(d_model) 使嵌入值与位置编码在同量级
    """
    def __init__(self, vocab_size: int, d_model: int):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        # x: (batch, seq_len)  ->  (batch, seq_len, d_model)
        return self.embedding(x) * math.sqrt(self.d_model)


# 快速验证
emb = InputEmbedding(vocab_size=100, d_model=16)
x = torch.randint(0, 100, (2, 5))          # batch=2, seq=5
print('输入 shape:', x.shape)
print('嵌入输出 shape:', emb(x).shape)     # (2, 5, 16)

## 3. 位置编码 (Positional Encoding)

Transformer 没有循环/卷积结构，需要额外注入位置信息：

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{\,2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{\,2i/d_{model}}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # 预计算位置编码矩阵 (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()          # (max_len, 1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )                                                              # (d_model/2,)
        pe[:, 0::2] = torch.sin(pos * div)   # 偶数列
        pe[:, 1::2] = torch.cos(pos * div)   # 奇数列

        # 注册为 buffer（不参与梯度更新，但随模型保存）
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# 可视化位置编码
pe_module = PositionalEncoding(d_model=64, dropout=0.0)
pe_matrix = pe_module.pe[0].numpy()   # (max_len, 64)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pe_matrix[:50, :], aspect='auto', cmap='RdBu')
plt.colorbar(im, ax=ax)
ax.set_xlabel('Embedding Dimension')
ax.set_ylabel('Position')
ax.set_title('Positional Encoding Matrix (first 50 positions, d_model=64)')
plt.tight_layout()
plt.show()

## 4. 缩放点积注意力 (Scaled Dot-Product Attention)

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

- **Q** (Query)：当前位置要查询什么信息
- **K** (Key)：每个位置提供的索引
- **V** (Value)：每个位置实际携带的内容
- 除以 $\sqrt{d_k}$ 防止点积值过大导致梯度消失

In [ ]:
def scaled_dot_product_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    mask: torch.Tensor = None
):
    """
    Args:
        Q: (batch, heads, seq_q, d_k)
        K: (batch, heads, seq_k, d_k)
        V: (batch, heads, seq_k, d_v)
        mask: (batch, 1, seq_q, seq_k)  —— 0 处将被 -inf 填充
    Returns:
        output: (batch, heads, seq_q, d_v)
        attn_weights: (batch, heads, seq_q, seq_k)
    """
    d_k = Q.size(-1)
    # 注意力得分矩阵
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (..., seq_q, seq_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)
    # nan 保护（全-inf 行 softmax 结果为 nan）
    attn_weights = torch.nan_to_num(attn_weights, nan=0.0)

    output = torch.matmul(attn_weights, V)
    return output, attn_weights


# 演示：3个词，d_k=4
Q = torch.randn(1, 1, 3, 4)
K = torch.randn(1, 1, 3, 4)
V = torch.randn(1, 1, 3, 4)
out, attn = scaled_dot_product_attention(Q, K, V)
print('输出 shape:', out.shape)        # (1,1,3,4)
print('注意力权重:\n', attn[0, 0].detach().numpy().round(3))

## 5. 多头注意力 (Multi-Head Attention)

$$\text{MultiHead}(Q, K, V) = \text{Concat}(head_1, \ldots, head_h)\,W^O$$
$$head_i = \text{Attention}(QW_i^Q,\; KW_i^K,\; VW_i^V)$$

多头机制让模型能**同时**关注来自不同表示子空间的信息。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model 必须能被 num_heads 整除'
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # 4 个线性投影：Q、K、V 和输出
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)
        self.attn_weights = None   # 保存供可视化用

    def split_heads(self, x):
        """(batch, seq, d_model) -> (batch, heads, seq, d_k)"""
        batch, seq, _ = x.size()
        return x.view(batch, seq, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        batch = query.size(0)

        # 线性投影 + 分头
        Q = self.split_heads(self.W_q(query))   # (batch, heads, seq_q, d_k)
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))

        # 缩放点积注意力
        x, self.attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        # 合并多头 (batch, seq_q, d_model)
        x = x.transpose(1, 2).contiguous().view(batch, -1, self.d_model)

        return self.W_o(x)


# 验证
mha = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)   # batch=2, seq=10, d_model=64
print('MHA 输出 shape:', mha(x, x, x).shape)   # (2, 10, 64)

## 6. 前馈网络 (Position-wise Feed-Forward Network)

$$\text{FFN}(x) = \max(0,\, xW_1 + b_1)\,W_2 + b_2$$

对序列中每个位置独立应用两层 MLP，中间维度通常是 `d_model` 的 4 倍。

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


ff = FeedForward(d_model=64, d_ff=256)
print('FFN 输出 shape:', ff(torch.randn(2, 10, 64)).shape)  # (2, 10, 64)

## 7. 残差连接 + 层归一化 (Add & LayerNorm)

每个子层输出为：$\text{LayerNorm}(x + \text{Sublayer}(x))$

残差连接缓解梯度消失；层归一化稳定训练。

In [ ]:
class AddNorm(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer_output):
        # Pre-LN 变体也很常用: x + sublayer(norm(x))
        return self.norm(x + self.dropout(sublayer_output))


add_norm = AddNorm(d_model=64)
x = torch.randn(2, 10, 64)
sub_out = torch.randn(2, 10, 64)
print('Add&Norm 输出 shape:', add_norm(x, sub_out).shape)

## 8. 编码器层 (Encoder Layer)

每个编码器层包含两个子层：
1. 多头自注意力 (Self-Attention)
2. 前馈网络 (FFN)

每个子层均有残差连接和层归一化。

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)

    def forward(self, x, src_mask=None):
        # 子层 1：自注意力
        x = self.add_norm1(x, self.self_attn(x, x, x, src_mask))
        # 子层 2：前馈网络
        x = self.add_norm2(x, self.ff(x))
        return x


class Encoder(nn.Module):
    def __init__(self, layer: EncoderLayer, num_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(layer.self_attn.d_model)

    def forward(self, x, src_mask=None):
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)


enc_layer = EncoderLayer(d_model=64, num_heads=8, d_ff=256)
encoder = Encoder(enc_layer, num_layers=6)
src = torch.randn(2, 10, 64)
enc_out = encoder(src)
print('编码器输出 shape:', enc_out.shape)   # (2, 10, 64)

## 9. 解码器层 (Decoder Layer)

每个解码器层包含**三个**子层：
1. **掩码多头自注意力** (Masked Self-Attention) — 因果掩码，防止看到未来 token
2. **交叉注意力** (Cross-Attention) — Q 来自解码器，K/V 来自编码器
3. 前馈网络 (FFN)

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)  # 掩码自注意力
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)  # 交叉注意力
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # 子层 1：因果自注意力
        x = self.add_norm1(x, self.self_attn(x, x, x, tgt_mask))
        # 子层 2：交叉注意力 (Q=解码器, K=V=编码器)
        x = self.add_norm2(x, self.cross_attn(x, enc_output, enc_output, src_mask))
        # 子层 3：前馈网络
        x = self.add_norm3(x, self.ff(x))
        return x


class Decoder(nn.Module):
    def __init__(self, layer: DecoderLayer, num_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(layer.self_attn.d_model)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return self.norm(x)


dec_layer = DecoderLayer(d_model=64, num_heads=8, d_ff=256)
decoder = Decoder(dec_layer, num_layers=6)
tgt = torch.randn(2, 7, 64)
dec_out = decoder(tgt, enc_out)
print('解码器输出 shape:', dec_out.shape)   # (2, 7, 64)

## 10. 完整 Transformer 模型

整合所有组件，构建完整的 Encoder-Decoder Transformer。

In [ ]:
def make_src_mask(src, pad_idx=0):
    """编码器 padding 掩码: 非 pad 位置为 1"""
    return (src != pad_idx).unsqueeze(1).unsqueeze(2)   # (batch, 1, 1, seq)


def make_tgt_mask(tgt, pad_idx=0):
    """解码器因果掩码 = padding 掩码 AND 下三角掩码"""
    batch, seq = tgt.size()
    pad_mask  = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)               # (batch, 1, 1, seq)
    causal    = torch.tril(torch.ones(seq, seq, device=tgt.device)).bool()  # (seq, seq)
    return pad_mask & causal.unsqueeze(0).unsqueeze(0)                   # (batch, 1, seq, seq)


class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int = 512,
        num_heads: int = 8,
        num_encoder_layers: int = 6,
        num_decoder_layers: int = 6,
        d_ff: int = 2048,
        max_len: int = 512,
        dropout: float = 0.1,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.pad_idx = pad_idx

        # 嵌入层
        self.src_emb = InputEmbedding(src_vocab_size, d_model)
        self.tgt_emb = InputEmbedding(tgt_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)

        # 编码器 & 解码器
        enc_layer = EncoderLayer(d_model, num_heads, d_ff, dropout)
        dec_layer = DecoderLayer(d_model, num_heads, d_ff, dropout)
        self.encoder = Encoder(enc_layer, num_encoder_layers)
        self.decoder = Decoder(dec_layer, num_decoder_layers)

        # 输出投影
        self.projection = nn.Linear(d_model, tgt_vocab_size)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src):
        src_mask = make_src_mask(src, self.pad_idx)
        x = self.pos_enc(self.src_emb(src))
        return self.encoder(x, src_mask), src_mask

    def decode(self, tgt, enc_output, src_mask):
        tgt_mask = make_tgt_mask(tgt, self.pad_idx)
        x = self.pos_enc(self.tgt_emb(tgt))
        return self.decoder(x, enc_output, src_mask, tgt_mask)

    def forward(self, src, tgt):
        enc_output, src_mask = self.encode(src)
        dec_output = self.decode(tgt, enc_output, src_mask)
        return self.projection(dec_output)   # (batch, tgt_seq, tgt_vocab)


# 统计参数量
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model = Transformer(
    src_vocab_size=200, tgt_vocab_size=200,
    d_model=128, num_heads=8,
    num_encoder_layers=3, num_decoder_layers=3,
    d_ff=512, dropout=0.1
).to(DEVICE)

print(f'模型总参数量: {count_params(model):,}')

# 前向传播验证
src_ids = torch.randint(1, 200, (2, 12)).to(DEVICE)
tgt_ids = torch.randint(1, 200, (2, 9)).to(DEVICE)
logits = model(src_ids, tgt_ids)
print('Logits shape:', logits.shape)   # (2, 9, 200)

## 11. 训练示例：数字序列翻转任务

任务：输入 `[3, 1, 4, 1, 5]`，输出 `[5, 1, 4, 1, 3]`（翻转）。

这是验证 seq2seq 模型是否正确学习序列变换的经典小任务。

In [ ]:
# ─── 超参 ───────────────────────────────────────────────
PAD_IDX = 0
BOS_IDX = 1   # Begin Of Sequence
EOS_IDX = 2   # End Of Sequence
VOCAB_SIZE = 13   # 0~12（0=PAD, 1=BOS, 2=EOS, 3~12=数字）
SEQ_LEN = 8
BATCH_SIZE = 64
N_TRAIN = 2000
N_EPOCHS = 20
LR = 1e-3


# ─── 数据集 ─────────────────────────────────────────────
class ReverseDataset(Dataset):
    """生成随机数字序列，标签为其翻转"""
    def __init__(self, n_samples, seq_len, vocab_size):
        self.data = torch.randint(3, vocab_size, (n_samples, seq_len))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]
        # src: BOS + seq
        src = torch.cat([torch.tensor([BOS_IDX]), seq])
        # tgt_in:  BOS + reversed(seq)
        # tgt_out: reversed(seq) + EOS
        rev = seq.flip(0)
        tgt_in  = torch.cat([torch.tensor([BOS_IDX]), rev])
        tgt_out = torch.cat([rev, torch.tensor([EOS_IDX])])
        return src, tgt_in, tgt_out


train_set = ReverseDataset(N_TRAIN, SEQ_LEN, VOCAB_SIZE)
val_set   = ReverseDataset(400,     SEQ_LEN, VOCAB_SIZE)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE)

print('训练集大小:', len(train_set))
src_ex, ti_ex, to_ex = train_set[0]
print('示例 src:    ', src_ex.tolist())
print('示例 tgt_in: ', ti_ex.tolist())
print('示例 tgt_out:', to_ex.tolist())

In [ ]:
# ─── 模型、损失、优化器 ─────────────────────────────────
model = Transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=64,
    num_heads=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    d_ff=256,
    dropout=0.1,
    pad_idx=PAD_IDX,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(train_loader), epochs=N_EPOCHS
)

print(f'模型参数量: {count_params(model):,}')

In [ ]:
# ─── 训练循环 ────────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, scheduler):
    model.train()
    total_loss = 0
    for src, tgt_in, tgt_out in loader:
        src, tgt_in, tgt_out = src.to(DEVICE), tgt_in.to(DEVICE), tgt_out.to(DEVICE)
        logits = model(src, tgt_in)                    # (batch, seq, vocab)
        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            tgt_out.reshape(-1)
        )
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = total = 0
    with torch.no_grad():
        for src, tgt_in, tgt_out in loader:
            src, tgt_in, tgt_out = src.to(DEVICE), tgt_in.to(DEVICE), tgt_out.to(DEVICE)
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
            total_loss += loss.item()
            preds = logits.argmax(dim=-1)           # (batch, seq)
            mask = tgt_out != PAD_IDX
            correct += (preds[mask] == tgt_out[mask]).sum().item()
            total   += mask.sum().item()
    return total_loss / len(loader), correct / total


train_losses, val_losses, val_accs = [], [], []

print(f'{"Epoch":>6} {"Train Loss":>12} {"Val Loss":>10} {"Val Acc":>10}')
print('-' * 44)

for epoch in range(1, N_EPOCHS + 1):
    tr_loss = train_epoch(model, train_loader, criterion, optimizer, scheduler)
    vl_loss, vl_acc = eval_epoch(model, val_loader, criterion)
    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    val_accs.append(vl_acc)
    if epoch % 2 == 0:
        print(f'{epoch:>6} {tr_loss:>12.4f} {vl_loss:>10.4f} {vl_acc:>9.1%}')

In [ ]:
# ─── 训练曲线 ────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Train Loss', linewidth=2)
ax1.plot(val_losses,   label='Val Loss',   linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Train / Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(val_accs, color='green', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Token Accuracy')
ax2.set_title('Validation Token Accuracy')
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))

plt.suptitle('Sequence Reversal Task — Training Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 自回归推理 ──────────────────────────────────────────
@torch.no_grad()
def translate(model, src_seq, max_len=20):
    """自回归逐步生成目标序列"""
    model.eval()
    src = torch.tensor([BOS_IDX] + src_seq).unsqueeze(0).to(DEVICE)
    enc_output, src_mask = model.encode(src)

    # 初始解码器输入：BOS
    tgt = torch.tensor([[BOS_IDX]], device=DEVICE)
    result = []

    for _ in range(max_len):
        dec_output = model.decode(tgt, enc_output, src_mask)
        next_token = model.projection(dec_output[:, -1]).argmax(-1).item()
        if next_token == EOS_IDX:
            break
        result.append(next_token)
        tgt = torch.cat([tgt, torch.tensor([[next_token]], device=DEVICE)], dim=1)

    return result


# 测试几个例子
print('\n推理测试 (数字范围 3~12):')
print(f'{"输入":>20} {"期望输出":>20} {"模型输出":>20} {"正确":>6}')
print('-' * 72)
for _ in range(6):
    seq = torch.randint(3, VOCAB_SIZE, (SEQ_LEN,)).tolist()
    expected = seq[::-1]
    predicted = translate(model, seq)
    ok = '✓' if predicted == expected else '✗'
    print(f'{str(seq):>20} {str(expected):>20} {str(predicted):>20} {ok:>6}')

## 12. 注意力权重可视化

提取编码器第一层各头的注意力权重，观察模型学到了什么。

In [ ]:
@torch.no_grad()
def get_attn_weights(model, src_seq):
    """返回编码器第一层所有头的注意力权重"""
    model.eval()
    src = torch.tensor([BOS_IDX] + src_seq).unsqueeze(0).to(DEVICE)
    src_mask = make_src_mask(src, PAD_IDX)
    x = model.pos_enc(model.src_emb(src))
    # 只过第一个编码器层以获取注意力
    first_layer = model.encoder.layers[0]
    first_layer.self_attn(x, x, x, src_mask)
    return first_layer.self_attn.attn_weights[0].cpu().numpy()  # (heads, seq, seq)


sample_seq = torch.randint(3, VOCAB_SIZE, (SEQ_LEN,)).tolist()
tokens = ['BOS'] + [str(t) for t in sample_seq]
attn_weights = get_attn_weights(model, sample_seq)   # (4 heads, 9, 9)

num_heads = attn_weights.shape[0]
fig, axes = plt.subplots(1, num_heads, figsize=(4 * num_heads, 4))

for h, ax in enumerate(axes):
    sns.heatmap(
        attn_weights[h], ax=ax,
        xticklabels=tokens, yticklabels=tokens,
        cmap='Blues', vmin=0, vmax=1,
        linewidths=0.3, linecolor='gray',
        cbar=(h == num_heads - 1)
    )
    ax.set_title(f'Head {h+1}', fontsize=11)
    ax.tick_params(labelsize=8)
    ax.set_xlabel('Key')
    if h == 0:
        ax.set_ylabel('Query')

plt.suptitle(f'Encoder Layer 1 Attention Weights\nInput: {tokens}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('attention_weights.png', dpi=120, bbox_inches='tight')
plt.show()

## 总结：Transformer 架构一览

| 组件 | 作用 | 关键公式 |
|------|------|----------|
| Input Embedding | token → 向量 | $xW_E \cdot \sqrt{d_{model}}$ |
| Positional Encoding | 注入位置信息 | $\sin/\cos(pos / 10000^{2i/d})$ |
| Scaled Dot-Product Attention | 计算注意力 | $\text{softmax}(QK^\top/\sqrt{d_k})V$ |
| Multi-Head Attention | 多视角注意力 | $h$ 个头并行，拼接后投影 |
| Feed-Forward Network | 逐位置变换 | 两层 MLP，中间维度 $d_{ff}$ |
| Add & LayerNorm | 稳定训练 | $\text{LayerNorm}(x + \text{Sublayer}(x))$ |
| Encoder | 理解源序列 | $N$ 层 (Self-Attn + FFN) |
| Decoder | 生成目标序列 | $N$ 层 (Masked Self-Attn + Cross-Attn + FFN) |

**推荐延伸阅读：**
- [原论文 Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- [Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/)